# 🦾 Módulo 01 — Introdução ao Isaac Lab

**Duração:** 30 minutos (0:25 – 0:55) | **Workshop Quadrúpede — Summit IA Joinville**

---

### 🎯 Objetivos deste módulo

1. ✅ Entender **o que é o Isaac Lab** e por que ele é especial
2. ✅ Compreender a **arquitetura do framework** (GPU → Sim → RL)
3. ✅ Conhecer os **componentes de um MDP** no Isaac Lab
4. ✅ Criar e executar seu **primeiro ambiente** (CartPole)
5. ✅ Estar pronto para o Módulo 02


## 🤔 O que é o Isaac Lab?

**Isaac Lab** é um framework open-source da NVIDIA para **treinamento de políticas de robótica com Reinforcement Learning** em simulação de alta fidelidade.

| Característica | Descrição |
|---|---|
| **GPU-Parallelism** | Roda **milhares** de ambientes simultâneos na mesma GPU |
| **Física realista** | Baseado em PhysX 5 / Newton |
| **Sensor rico** | Câmeras RGB-D, LiDAR, IMU — tudo simulado |
| **+100 ambientes** | Locomoção, manipulação, navegação... |
| **Multi-backend RL** | RSL-RL, skrl, Stable-Baselines3 |
| **Open Source** | Apache 2.0 |

### 📊 Eficiência GPU-parallel:
```
CPU tradicional:  1 ambiente × 1000 steps/s =     1,000 steps/s
Isaac Lab (GPU): 4096 ambientes × 250  steps/s > 1,000,000 steps/s  ← 1000x mais rápido!
```


## 🏗️ Arquitetura do Sistema

```
┌─────────────────────────────────────────────────┐
│              NVIDIA GPU (RTX/A-series)          │
│  ┌─────────────────────────────────────────┐   │
│  │         ISAAC SIM 5.1 (Omniverse)       │   │
│  │  PhysX 5 / Newton  |  RTX Rendering     │   │
│  └─────────────────────────────────────────┘   │
│  ┌─────────────────────────────────────────┐   │
│  │              ISAAC LAB                  │   │
│  │  Managers | Assets | Sensors | Terrain  │   │
│  └─────────────────────────────────────────┘   │
│  ┌─────────────────────────────────────────┐   │
│  │  RSL-RL (PPO) | skrl | SB3 | RL-Games   │   │
│  └─────────────────────────────────────────┘   │
└─────────────────────────────────────────────────┘
                        │
               🤖 Política de Controle
          (rede neural treinada → robô físico)
```


## 📊 Isaac Lab vs Outros Frameworks

| Característica | **Isaac Lab** | IsaacGym | MuJoCo | PyBullet |
|---|---|---|---|---|
| Motor de física | PhysX 5 / Newton | PhysX 4 | MuJoCo | Bullet |
| Parallelismo GPU | ✅ Nativo | ✅ Nativo | ❌ CPU | ❌ CPU |
| Sensores realistas | ✅ RGB-D, Lidar | ⚠️ Limitado | ❌ | ❌ |
| Ambientes prontos | +100 | ~30 | ~20 | ~15 |
| Open Source | ✅ Apache 2.0 | ✅ | ⚠️ | ✅ |
| Ativamente mantido | ✅ NVIDIA | ❌ Deprecated | ✅ | ⚠️ |

> ⚠️ **Importante:** IsaacGym foi **descontinuado** em 2024. Isaac Lab é seu sucessor oficial.


In [ ]:
# ✅ Verificar instalação do Isaac Lab
import sys

pacotes = [
    ('isaaclab',       'Isaac Lab core'),
    ('isaaclab_tasks', 'Isaac Lab tasks'),
    ('rsl_rl',         'RSL-RL PPO'),
    ('torch',          'PyTorch'),
    ('gymnasium',      'Gymnasium API'),
]

for pacote, descricao in pacotes:
    try:
        mod = __import__(pacote)
        versao = getattr(mod, '__version__', '?')
        print(f'  ✅ {pacote:<25} ({descricao}) v{versao}')
    except ImportError:
        print(f'  ❌ {pacote:<25} ({descricao}) - NÃO INSTALADO')


In [ ]:
# 📋 Listar ambientes de locomoção disponíveis
print('🤖 Ambientes de Locomoção no Isaac Lab')
print('=' * 60)

try:
    import gymnasium as gym
    import isaaclab_tasks  # registra todos os ambientes
    import workshop_quadrupede  # registra Workshop-Anymal-v0
    
    termos = ['Anymal', 'Locomotion', 'Quadruped', 'Workshop']
    encontrados = [tid for tid in gym.registry.keys()
                   if any(t.lower() in tid.lower() for t in termos)]
    
    for env_id in sorted(encontrados):
        marcador = '  🌟' if 'Workshop' in env_id else '    '
        print(f'{marcador} {env_id}')
    
    print(f'\nTotal registrados: {len(gym.registry)}')
except ImportError as e:
    print(f'⚠️  Isaac Lab não disponível: {e}')
    ambientes_exemplo = [
        'Isaac-Velocity-Flat-Anymal-C-v0',
        'Isaac-Velocity-Rough-Anymal-C-v0',
        'Isaac-Velocity-Flat-Spot-v0',
        'Workshop-Anymal-v0  ← Nosso ambiente!',
    ]
    print('\nAmbientes esperados:')
    for env in ambientes_exemplo:
        print(f'  • {env}')


## 🎡 A API Isaac Lab: CartPole

O fluxo de uso é sempre o mesmo — para o CartPole simples ou para o Anymal complexo:

```python
# 1. Inicializar Isaac Sim (SEMPRE primeiro)
from isaaclab.app import AppLauncher
app = AppLauncher(headless=True)
sim = app.app

# 2. Imports
import gymnasium as gym
import torch
import isaaclab_tasks  # registra ambientes

# 3. Criar ambiente (64 ambientes em paralelo!)
env = gym.make('Isaac-Cartpole-v0', num_envs=64, device='cuda')

# 4. Reset
obs, info = env.reset()
print(f'Obs shape: {obs["policy"].shape}')  # (64, 4)

# 5. Loop de simulação
for step in range(100):
    action = torch.rand(64, 1, device='cuda') * 2 - 1  # aleatório
    obs, reward, done, truncated, info = env.step(action)

env.close()
sim.close()
```

> 💡 **A API é idêntica** para o Anymal C — só muda o `env_id` e as dimensões!


In [ ]:
# 📊 Demonstração da estrutura do MDP
# (Valores simulados — o Isaac Sim real precisa rodar via CLI)
import numpy as np

print('=== Estrutura do MDP Isaac Lab ===')
print()

# CartPole
print('CartPole (simples):')
print(f'  Obs space:    Box(-inf, inf, (4,), float32)')
print(f'  Action space: Box(-1.0, 1.0, (1,), float32)')
print(f'  Num envs:     64')
print(f'  Obs shape:    (64, 4)')
print()

# Anymal C
print('Workshop-Anymal-v0 (nosso robô):')
print(f'  Obs space:    Box(-inf, inf, (48,), float32)')
print(f'  Action space: Box(-1.0, 1.0, (12,), float32)')
print(f'  Num envs:     4096 (treino) / 16 (debug)')
print(f'  Obs shape:    (4096, 48)')
print()
print('obs é sempre um DICT: {"policy": tensor}')
print('reward é tensor: shape=(num_envs,)')


## 🧩 Componentes de um MDP no Isaac Lab

```
                 oₜ (obs, 48D)
  ┌──────────┐ ──────────────▶ ┌──────────────┐
  │  Isaac   │                │   Política π  │
  │   Sim    │ ◀────────────── │  (Rede Neural)│
  └──────────┘  aₜ (ação, 12D) └──────────────┘
       │
  sₜ₊₁, rₜ, doneₜ
```

| Componente | Anymal C | Descrição |
|---|---|---|
| **Observation** | 48D | Tudo que o robô 'sente' |
| **Action** | 12D | Joint position targets |
| **Reward** | escalar | Seguir vel + estabilidade |
| **Done** | bool | Caiu ou timeout |
| **Info** | dict | Métricas extras (debug) |


In [ ]:
# 📊 Resumo do módulo 01
print('=' * 60)
print(' 📚 RESUMO — Módulo 01: Introdução ao Isaac Lab')
print('=' * 60)

conceitos = [
    ('Isaac Lab', 'Framework NVIDIA para RL em robótica GPU-parallel'),
    ('GPU-Parallel', '4096 ambientes simultâneos → 1000x mais rápido que CPU'),
    ('API Gymnasium', 'reset() → step(action) → (obs, reward, done, truncated, info)'),
    ('MDP', 'Obs → Ação → Reward → Próx Obs — ciclo fundamental do RL'),
    ('RSL-RL + PPO', 'Biblioteca ETH Zurich otimizada para GPU'),
]

for i, (conceito, desc) in enumerate(conceitos, 1):
    print(f'\n  {i}. 🔑 {conceito}')
    print(f'     {desc}')

print('\n' + '=' * 60)
print(' ➡️  Próximo: Módulo 02 — Ambiente do Quadrúpede Anymal C')
print('=' * 60)
print('\n⏰ Tempo: ~30 min | Restante: ~2h00min')
